In [ ]:
import pandas as pd
import logging
from openai import OpenAI
from ollama import Client

import time
import logging
import re

# Configuração do logger
# logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

ollama_models = [
    'deepseek-r1:8b',
    'llama3.1:8b',
    'llama3.2:3b',
    'phi3:14b',
    'qwen2.5:14b',
    'qwen2.5:7b',
    'gemma2:9b',
    'qwen2:7b',
    'gemma3:12b', 
]

gpt_models = ['gpt-4.1-mini', 'gpt-4o-mini']

class NewsProcessor:
    LABELS_VALIDOS = ["Verídico", "Inconclusivo", "Inverídico"]
    CATEGORIAS_VALIDAS = [
        "Fake News",
        "Descontextualização",
        "Discurso de ódio",
        "Extremismo antidemocrático",
        "Não se aplica"
    ]

    def __init__(self, model, host_name='', 
                 path_api_key='key_openai.txt'):
        self.model = model
        self.host_name = host_name

        if model in ollama_models:
            self.style = "ollama"
            self.client = Client(host=host_name)
        elif model in gpt_models:
            self.style = "openai"
            if path_api_key is None:
                raise ValueError("Você precisa fornecer o caminho para a chave da API OpenAI.")
            with open(path_api_key, 'r') as f:
                api_key = f.read().strip()
            self.client = OpenAI(api_key=api_key)
        else:
            raise ValueError(f"Modelo não reconhecido: {model}")

        self.tasks = {
            "veridica": "Liste apenas as afirmações VERÍDICAS da notícia. Responda de forma direta, apenas os tópicos.",
            "inveridica": "Liste apenas as afirmações INVERÍDICAS da notícia. Responda de forma direta, apenas os tópicos.",
            "inconclusivas": "Liste apenas as afirmações INCONCLUSIVAS da notícia. Responda de forma direta, apenas os tópicos.",
            "label": (
                "Qual o seu parecer quanto ao nível de veracidade do conteúdo?\n"
                "Sua resposta deve ser apenas uma das opções: Verídico, Inconclusivo ou Inverídico.\n"
            ),
            "categoria": (
                "Considerando todos os elementos identificados na mídia analisada, qual categoria melhor representa, "
                "de forma predominante, o aspecto que mais se destaca no conteúdo? Mesmo que outros elementos estejam presentes, "
                "informe apenas a opção mais evidente. As opções possíveis são: Fake News (divulgação de fatos sabidamente inverídicos, falsificações, boatos), "
                "Descontextualização (uso de informações verdadeiras fora de contexto...),"
                "Discurso de ódio (estímulo à violência física ou simbólica...), "
                "Extremismo antidemocrático (narrativas que promovem a ruptura institucional...), "
                "Não se aplica (Nenhum dos elementos). "
                "Sua resposta deve conter apenas uma dessas categorias, e nada mais."
            )
        }

    def _query_ollama(self, task, text):
        attempts = 0
        erro_504_count = 0

        while attempts < 5:
            try:
                response = self.client.chat(
                    model=self.model,
                    options={"temperature": 0.0},
                    messages=[
                        {
                            "role": "user",
                            "content": f"{task}\n===\n{text}",
                        },
                    ],
                )
                return response["message"]["content"].strip()
            
            except Exception as e:
                error_msg = str(e).lower()
                attempts += 1

                if "504" in error_msg or "gateway time-out" in error_msg or "504 gateway" in error_msg:
                    erro_504_count += 1
                    print(f"❌ Erro 504 na requisição Ollama (tentativa {attempts}/5). Aguardando 20s...")
                else:
                    print(f"⚠️ Erro inesperado na requisição Ollama: {e}. Aguardando 20s...")

                time.sleep(20)

        if erro_504_count == 5:
            print("🚫 Todas as tentativas falharam com erro 504. Retornando respostas vazias.")
            return ""  # ou return "SEM RESPOSTA", se preferir visibilidade no log

        # Se chegou aqui, falhou com outro erro (não só 504)
        self._reconnect_client()
        return self._query_ollama(task, text)

    def _query_openai(self, task, text):
        while True:
            try:
                response = self.client.chat.completions.create(
                    model=self.model.replace("-mini", ""),
                    messages=[
                        {
                            "role": "user",
                            "content": f"{task}\n===\n{text}",
                        },
                    ],
                    temperature=0.0,
                )
                return response.choices[0].message.content.strip()
            except Exception as e:
                print(f"Erro na requisição OpenAI: {e}. Tentando novamente em 20 segundos...")
                time.sleep(20)

    def query(self, task, text):
        if self.style == "ollama":
            return self._query_ollama(task, text)
        elif self.style == "openai":
            return self._query_openai(task, text)
        
    def generate_embedding(self, task, text):
        prompt = f"{task}\n===\n{text}"

        if self.style == "ollama":
            # Substitui o modelo apenas para embeddings se for gemma3:12b
            model_para_embedding = "nomic-embed-text:latest" if self.model == "gemma3:12b" else self.model

            response = self.client.embeddings(model=model_para_embedding, prompt=prompt)
            embedding = {
                "model": model_para_embedding,
                "style": "ollama",
                "embedding": response["embedding"],
                "prompt": prompt,
            }
            return embedding

        elif self.style == "openai":
            response = self.client.embeddings.create(
                model="text-embedding-3-small",
                input=prompt,
                encoding_format="float"
            )
            embedding = {
                "model": "text-embedding-3-small",
                "style": "openai",
                "embedding": response.data[0].embedding,
                "prompt": prompt,
            }
            return embedding

    def justify(self, dataframe):
        transcricoes = dataframe[["start", "speaker", "text"]].drop_duplicates().copy()
        prompt_intro = (
            "Para sua resposta, aja como um fact-checker sênior avaliando uma notícia enviada em formato de vídeo. "
            "A seguir está a transcrição detalhada do conteúdo, incluindo falas e trechos informativos."
        )
        prompt_body = "\n\n".join(
            f"[{row['start']}] {row['speaker']}: {row['text']}" for _, row in transcricoes.iterrows()
        )
        full_text = f"{prompt_intro}\n\n{prompt_body}"
        return full_text

    def verificar_e_corrigir_resposta(self, texto_noticia, task, opcoes_validas, max_tentativas=4):
        inicio = time.time()

        regex_opcoes = [re.compile(rf"\b{re.escape(opcao)}\b", re.IGNORECASE) for opcao in opcoes_validas]

        ultima_resposta = None
        embeddings = None

        for tentativa in range(1, max_tentativas + 1):
            resposta = self.query(self.tasks[task], texto_noticia).strip()
            embeddings = self.generate_embedding(self.tasks[task], texto_noticia)
            ultima_resposta = resposta

            if "deepseek" in self.model.lower():
                ultima_pos = -1
                ultima_opcao = None
                for padrao, opcao in zip(regex_opcoes, opcoes_validas):
                    matches = list(padrao.finditer(resposta))
                    if matches and matches[-1].start() > ultima_pos:
                        ultima_pos = matches[-1].start()
                        ultima_opcao = opcao
                if ultima_opcao:
                    return ultima_opcao, embeddings

            else:
                for padrao, opcao in zip(regex_opcoes, opcoes_validas):
                    if padrao.search(resposta):
                        return opcao, embeddings

        duracao = round(time.time() - inicio, 2)
        logging.error(
            f"❌ Falha ao obter resposta válida para '{task}' após {max_tentativas} tentativas. "
            f"Duração total: {duracao}s. Última resposta: '{ultima_resposta}'"
        )
        return ultima_resposta, embeddings


    def answer_generate(self, dataframe):
        texto_noticia = self.justify(dataframe)

        dados = {}
        for task in ['veridica', 'inveridica', 'inconclusivas']:
            dados[task] = self.query(self.tasks[task], texto_noticia)

        dados['label'], embeddings_label = self.verificar_e_corrigir_resposta(texto_noticia, 'label', self.LABELS_VALIDOS)
        dados['categoria'], embeddings_category = self.verificar_e_corrigir_resposta(texto_noticia, 'categoria', self.CATEGORIAS_VALIDAS)

        justification_prompt = (
            "Com base nas informações extraídas da análise da notícia abaixo, elabore uma justificativa clara e objetiva "
            "(máximo 300 palavras) para o parecer e categoria finais atribuídos.\n\n"
            f"- Afirmações verídicas identificadas:\n{dados['veridica']}\n\n"
            f"- Afirmações inverídicas identificadas:\n{dados['inveridica']}\n\n"
            f"- Afirmações inconclusivas identificadas:\n{dados['inconclusivas']}\n\n"
            f"- A categoria da notícia: {dados['categoria']}\n\n"
            f"- Parecer final atribuído (label): {dados['label']}\n\n"
            "\n\n"
        )
        dados['justification'] = self.query(justification_prompt, texto_noticia)
        return dados, embeddings_label['embedding'], embeddings_category['embedding']

In [ ]:
from collections import defaultdict
import pickle
import pandas as pd
import glob
import os

models = [
    # 'llama3.2-vision:11b', 
    # 'qwen2.5vl:7b', 
    "gemma3:12b", 
    # 'deepseek-r1:8b',
    # 'llama3.1:8b', 
    # 'llama3.2:3b', 
    'phi3:14b', 
    'qwen2.5:14b', 
    # 'qwen2.5:7b', 
    # 'deepseek-r1:32b', 
    # 'gemma2:9b', 
    # 'qwen2:7b',    
    # 'gpt-4.1-mini',
    # 'gpt-4o-mini'
]


def carregar_existentes(nome_base, model, root_dir="results_llm"):
    """
    Verifica se já existe um arquivo de saída para o modelo e nome_base.
    Se existir, retorna o DataFrame existente e o conjunto de video_ids processados.
    """
    pasta_saida = os.path.join(root_dir, model.replace(':', '-'))
    os.makedirs(pasta_saida, exist_ok=True)

    nome_saida = os.path.join(pasta_saida, f"{nome_base}_results.pkl")

    if os.path.exists(nome_saida):
        df_existente = pd.read_pickle(nome_saida)
        video_ids_processados = set(df_existente['video_id'].unique())
        print(f"⚠️  {len(video_ids_processados)} vídeos já processados serão ignorados.")
    else:
        df_existente = pd.DataFrame()
        video_ids_processados = set()

    return nome_saida, df_existente, video_ids_processados

def print_progress_bar(completed, total, prefix='', length=30):
    percent = completed / total
    filled = int(length * percent)
    bar = '█' * filled + '-' * (length - filled)
    print(f"\r{prefix} |{bar}| {int(percent * 100)}% concluído", end='')

def salvar_embeddings(embeddings_dict, tipo, nome_base):
    if embeddings_dict.empty:
        print("⚠️ Nenhum embedding para salvar.")
        return

    pasta_saida = os.path.join("results_llm", "embeddings", tipo)
    os.makedirs(pasta_saida, exist_ok=True)
    caminho = os.path.join(pasta_saida, f"{nome_base}_embeddings_{tipo}.pkl")
    
    embeddings_dict.to_pickle(caminho)
    # print(embeddings_dict.columns)
    print(f"📦 Embeddings filtrados e salvos: {caminho}")


def carregar_embeddings_existentes(tipo, nome_base):
    pasta_saida = os.path.join("results_llm", "embeddings", tipo)
    caminho_pkl = os.path.join(pasta_saida, f"{nome_base}_embeddings_{tipo}.pkl")

    if os.path.exists(caminho_pkl):
        df = pd.read_pickle(caminho_pkl)

        if df.index.name != "video_id":
            df.index.name = "video_id"
        df.index = df.index.astype(int)

        # Garante que a coluna 'video_id' também exista
        if "video_id" not in df.columns:
            df["video_id"] = df.index

        print(f"📂 Embeddings '{tipo}' carregados de: {caminho_pkl}")
        return df

    # Retorna DataFrame vazio com índice 'video_id' e a coluna presente
    df_vazio = pd.DataFrame(columns=["video_id"]).set_index("video_id")
    return df_vazio


arquivos_csv = glob.glob(r'working/results_whisper_*.csv')
for arquivo_path in arquivos_csv:
    nome_base = os.path.basename(arquivo_path).replace("results_whisper_", "").replace(".csv", "")
    df = pd.read_csv(arquivo_path)

    embeddings_por_video_label = carregar_embeddings_existentes("label", nome_base)
    embeddings_por_video_category = carregar_embeddings_existentes("category", nome_base)

    print(type(embeddings_por_video_category), embeddings_por_video_category.shape, embeddings_por_video_category.columns)

    for model in models:
        print(f"\n🔍 Processando arquivo '{nome_base}' com modelo '{model}'...")

        processor = NewsProcessor(
            model=model,
            host_name="",
            path_api_key='key_openai.txt'
        )

        resultados_modelo = []
        total_videos = df['video_id'].nunique()

        nome_saida, df_existente, video_ids_processados = carregar_existentes(nome_base, model)

        embedding_temp_label = pd.DataFrame()
        embedding_temp_category = pd.DataFrame()

        model_key = model.replace(':', '-')
        path_temp_label = f"temp_label_{model_key}_{nome_base}.pkl"
        path_temp_category = f"temp_category_{model_key}_{nome_base}.pkl"
        path_temp_resultado = f"temp_resultado_{model_key}_{nome_base}.pkl"

        # Inicializa listas ou carrega se já existirem
        def carregar_temp(caminho):
            if os.path.exists(caminho):
                with open(caminho, 'rb') as f:
                    return pickle.load(f)
            return []

        lista_temp_label = carregar_temp(path_temp_label)
        lista_temp_category = carregar_temp(path_temp_category)
        resultados_modelo = carregar_temp(path_temp_resultado)

        for i, video_id in enumerate(df['video_id'].unique(), start=1):
            if video_id in video_ids_processados:
                print(f"⏭️ Pulando vídeo {video_id}, já processado para modelo '{model}'")
                continue

            df_video = df[df['video_id'] == video_id][['start', 'speaker', 'text']].copy()
            if df_video.empty:
                continue

            try:
                inicio_tempo = time.time()
                resultado, embeddings_label, embeddings_category = processor.answer_generate(df_video)
                tempo_total = round(time.time() - inicio_tempo, 2)

                resultado['video_id'] = video_id
                resultado['tempo_processamento_seg'] = tempo_total
                resultados_modelo.append(resultado)

                # Armazena os embeddings em dicionários temporários
                lista_temp_label.append({
                    "video_id": int(video_id),
                    "nome_base": nome_base,
                    f"embedding_{model.replace(':', '-')}": embeddings_label
                })

                lista_temp_category.append({
                    "video_id": int(video_id),
                    "nome_base": nome_base,
                    f"embedding_{model.replace(':', '-')}": embeddings_category
                })

                # Salvar arquivos temporários imediatamente após o processamento
                with open(path_temp_resultado, 'wb') as f:
                    pickle.dump(resultados_modelo, f)

                with open(path_temp_label, 'wb') as f:
                    pickle.dump(lista_temp_label, f)

                with open(path_temp_category, 'wb') as f:
                    pickle.dump(lista_temp_category, f)
                # print(resultados_modelo, lista_temp_category, lista_temp_label)
                print_progress_bar(i, total_videos, prefix=f"Modelo {model}")

                print_progress_bar(i, total_videos, prefix=f"Modelo {model}")

            except Exception as e:
                print(f"\n❌ Erro ao processar vídeo {video_id} com modelo {model}: {e}")
        
        if resultados_modelo:
            df_novos = pd.DataFrame(resultados_modelo)
            df_final = pd.concat([df_existente, df_novos], ignore_index=True).drop_duplicates(subset=["video_id"])
            df_final.to_pickle(nome_saida)
            print(f"\n💾 Resultado salvo em: {nome_saida}")

        # ✅ Após processar todos os vídeos, cria DataFrames dos embeddings
        df_embedding_label = pd.DataFrame(lista_temp_label)
        df_embedding_category = pd.DataFrame(lista_temp_category)

        if not df_embedding_label.empty:
            df_embedding_label['video_id'] = df_embedding_label['video_id'].astype(int)
            df_embedding_label['nome_base'] = df_embedding_label['nome_base'].astype(str)
            df_embedding_label = df_embedding_label.reset_index(drop=True)

            if embeddings_por_video_label.empty:
                salvar_embeddings(df_embedding_label, "label", nome_base)
            else:
                embeddings_por_video_label['video_id'] = embeddings_por_video_label['video_id'].astype(int)
                embeddings_por_video_label['nome_base'] = embeddings_por_video_label['nome_base'].astype(str)
                embeddings_por_video_label = embeddings_por_video_label.reset_index(drop=True)

                embeddings_por_video_label = pd.merge(
                    embeddings_por_video_label,
                    df_embedding_label,
                    on=['video_id', 'nome_base'],
                    how='outer'
                )

                salvar_embeddings(embeddings_por_video_label, "label", nome_base)

        if not df_embedding_category.empty:
            df_embedding_category['video_id'] = df_embedding_category['video_id'].astype(int)
            df_embedding_category['nome_base'] = df_embedding_category['nome_base'].astype(str)
            df_embedding_category = df_embedding_category.reset_index(drop=True)

            if embeddings_por_video_category.empty:
                salvar_embeddings(df_embedding_category, "category", nome_base)
            else:
                embeddings_por_video_category['video_id'] = embeddings_por_video_category['video_id'].astype(int)
                embeddings_por_video_category['nome_base'] = embeddings_por_video_category['nome_base'].astype(str)
                embeddings_por_video_category = embeddings_por_video_category.reset_index(drop=True)

                embeddings_por_video_category = pd.merge(
                    embeddings_por_video_category,
                    df_embedding_category,
                    on=['video_id', 'nome_base'],
                    how='outer'
                )

                salvar_embeddings(embeddings_por_video_category, "category", nome_base)


📂 Embeddings 'label' carregados de: results_llm\embeddings\label\large-v3_embeddings_label.pkl
📂 Embeddings 'category' carregados de: results_llm\embeddings\category\large-v3_embeddings_category.pkl
<class 'pandas.core.frame.DataFrame'> (119, 4) Index(['video_id', 'nome_base', 'embedding_phi3-14b', 'embedding_gemma3-12b'], dtype='object')

🔍 Processando arquivo 'large-v3' com modelo 'gemma3:12b'...
⚠️  119 vídeos já processados serão ignorados.
⏭️ Pulando vídeo 28, já processado para modelo 'gemma3:12b'
⏭️ Pulando vídeo 14, já processado para modelo 'gemma3:12b'
⏭️ Pulando vídeo 33, já processado para modelo 'gemma3:12b'
⏭️ Pulando vídeo 113, já processado para modelo 'gemma3:12b'
⏭️ Pulando vídeo 127, já processado para modelo 'gemma3:12b'
⏭️ Pulando vídeo 40, já processado para modelo 'gemma3:12b'
⏭️ Pulando vídeo 139, já processado para modelo 'gemma3:12b'
⏭️ Pulando vídeo 19, já processado para modelo 'gemma3:12b'
⏭️ Pulando vídeo 17, já processado para modelo 'gemma3:12b'
⏭️ Pulan